# GPU Matrix Addition & Subtraction

Runs on Google Colab. Go to **Runtime → Change runtime type → T4 GPU** before running.

- Performs matrix **addition** and **subtraction** on the GPU using PyTorch (pre-installed on Colab)
- Verifies correctness against NumPy on the CPU
- Includes a small hand-verifiable example

In [ ]:
import torch
import numpy as np

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    device = torch.device('cpu')
    print("No GPU found — running on CPU. Go to Runtime → Change runtime type → T4 GPU.")

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")

## 1. Create Matrices

In [ ]:
np.random.seed(42)

SIZE = 1024
A_cpu = np.random.rand(SIZE, SIZE).astype(np.float32)
B_cpu = np.random.rand(SIZE, SIZE).astype(np.float32)

# Upload to GPU
A_gpu = torch.tensor(A_cpu, device=device)
B_gpu = torch.tensor(B_cpu, device=device)

print(f"Matrix A: {A_gpu.shape}, dtype={A_gpu.dtype}, device={A_gpu.device}")
print(f"Matrix B: {B_gpu.shape}, dtype={B_gpu.dtype}, device={B_gpu.device}")

## 2. GPU Addition & Subtraction

In [ ]:
import time

def timed_op(fn):
    """Run fn, synchronize if on CUDA, return (result, elapsed_ms)."""
    if device.type == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()
    result = fn()
    if device.type == 'cuda':
        torch.cuda.synchronize()
    return result, (time.perf_counter() - start) * 1000

# Warm-up
_ = A_gpu + B_gpu
_ = A_gpu - B_gpu

C_add_gpu, add_time = timed_op(lambda: A_gpu + B_gpu)
C_sub_gpu, sub_time = timed_op(lambda: A_gpu - B_gpu)

print(f"GPU addition:     {add_time:.2f} ms  result shape: {tuple(C_add_gpu.shape)}")
print(f"GPU subtraction:  {sub_time:.2f} ms  result shape: {tuple(C_sub_gpu.shape)}")

## 3. CPU Verification (NumPy)

In [ ]:
start = time.perf_counter()
C_add_cpu = A_cpu + B_cpu
cpu_add_time = (time.perf_counter() - start) * 1000

start = time.perf_counter()
C_sub_cpu = A_cpu - B_cpu
cpu_sub_time = (time.perf_counter() - start) * 1000

print(f"CPU addition:     {cpu_add_time:.2f} ms")
print(f"CPU subtraction:  {cpu_sub_time:.2f} ms")
print(f"\nAddition speedup:    {cpu_add_time/add_time:.1f}x")
print(f"Subtraction speedup: {cpu_sub_time/sub_time:.1f}x")

## 4. Correctness Check

In [ ]:
add_gpu_np = C_add_gpu.cpu().numpy()
sub_gpu_np = C_sub_gpu.cpu().numpy()

tol = 1e-5  # float32 element-wise ops are exact to machine precision

add_match = np.allclose(add_gpu_np, C_add_cpu, atol=tol)
sub_match = np.allclose(sub_gpu_np, C_sub_cpu, atol=tol)

print("=== Correctness Check ===")
print(f"Addition    — max diff: {np.max(np.abs(add_gpu_np - C_add_cpu)):.2e}  {'PASS' if add_match else 'FAIL'}")
print(f"Subtraction — max diff: {np.max(np.abs(sub_gpu_np - C_sub_cpu)):.2e}  {'PASS' if sub_match else 'FAIL'}")

print("\n=== Spot Check (top-left 3x3) ===")
print("GPU A+B:\n", add_gpu_np[:3, :3])
print("CPU A+B:\n", C_add_cpu[:3, :3])
print("GPU A-B:\n", sub_gpu_np[:3, :3])
print("CPU A-B:\n", C_sub_cpu[:3, :3])

## 5. Hand-Verifiable Example

Small matrices you can check manually:

```
A = [[1, 2],   B = [[5, 6],
     [3, 4]]        [7, 8]]

A + B = [[ 6,  8],    A - B = [[-4, -4],
         [10, 12]]              [-4, -4]]
```

In [ ]:
A_s = np.array([[1, 2], [3, 4]], dtype=np.float32)
B_s = np.array([[5, 6], [7, 8]], dtype=np.float32)

A_st = torch.tensor(A_s, device=device)
B_st = torch.tensor(B_s, device=device)

add_expected = np.array([[6, 8], [10, 12]], dtype=np.float32)
sub_expected = np.array([[-4, -4], [-4, -4]], dtype=np.float32)

gpu_add = (A_st + B_st).cpu().numpy()
gpu_sub = (A_st - B_st).cpu().numpy()
cpu_add = A_s + B_s
cpu_sub = A_s - B_s

print("A + B:")
print(f"  Expected: {add_expected.tolist()}")
print(f"  GPU:      {gpu_add.tolist()}")
print(f"  CPU:      {cpu_add.tolist()}")
print(f"  Match:    {'PASS' if np.array_equal(gpu_add, add_expected) else 'FAIL'}")

print("\nA - B:")
print(f"  Expected: {sub_expected.tolist()}")
print(f"  GPU:      {gpu_sub.tolist()}")
print(f"  CPU:      {cpu_sub.tolist()}")
print(f"  Match:    {'PASS' if np.array_equal(gpu_sub, sub_expected) else 'FAIL'}")